In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

c:\Users\Sudhe\anaconda3\envs\sudheer_agentic_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [5]:
embeddings.embed_query("Sudheer Gundra")

[0.010436558,
 0.008574519,
 0.0003719317,
 -0.069236964,
 0.0077735456,
 -0.0037097712,
 0.0055875382,
 0.0069977865,
 -0.043152805,
 0.0046500335,
 0.00023631947,
 -0.011199262,
 0.012905614,
 0.017075852,
 0.10614424,
 -0.024858518,
 -0.007957793,
 -0.0068158284,
 -0.0018409678,
 0.000787462,
 -0.0067347363,
 -0.0041993824,
 -0.007840963,
 -0.0023459736,
 -0.0020168142,
 0.0073404885,
 0.04371011,
 -0.019291263,
 0.02578254,
 0.005108317,
 0.0017128895,
 -0.022340385,
 0.01914564,
 0.014022501,
 0.027790463,
 -0.003167328,
 -0.004425969,
 0.008833405,
 -0.017211992,
 0.011941825,
 -0.0022798942,
 0.006855258,
 0.002754216,
 0.005376766,
 0.011033177,
 -0.0013423762,
 -0.0070556873,
 -0.011311683,
 0.009724292,
 0.014249221,
 0.002653204,
 -0.008671388,
 -0.00397634,
 -0.16703445,
 -0.01783067,
 -0.011380168,
 0.01838562,
 -0.002626922,
 0.017339168,
 0.013863482,
 -0.01770985,
 -0.005314204,
 -0.008113854,
 0.0041604373,
 0.006375443,
 0.0036764874,
 0.0064424374,
 -0.0036412654,
 -

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
documents=["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."]

In [8]:
my_query  = "Who is a prime minister of India?"

In [9]:
embedded_docs = embeddings.embed_documents(documents)
embedded_query = embeddings.embed_query(my_query)

In [10]:
cosine_similarity([embedded_query], embedded_docs)

array([[0.55572604, 0.61247309, 0.77749546]])

In [11]:
from sklearn.metrics.pairwise import euclidean_distances

In [12]:
euclidean_distances([embedded_query], embedded_docs)

array([[0.9426282 , 0.88037141, 0.66709004]])

In [14]:
from sklearn.metrics.pairwise import linear_kernel
import numpy as np

documents1 = [
    "Washington is the capital of USA",
    "Donald Trump is a president of USA",
    "Narendra Modi is a prime minister of India."
]

my_query1 = "Who is a prime minister of India?"


# Generate embeddings
embedded_docs1 = embeddings.embed_documents(documents1)

embedded_query1 = embeddings.embed_query(my_query1)


# Convert query to 2D
embedded_query1 = np.array(
    embedded_query1
).reshape(1, -1)


# Dot product similarity
scores = linear_kernel(
    embedded_query1,
    embedded_docs1
)


print(scores)

[[0.55572604 0.61247308 0.77749551]]


In [15]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

C:\Users\Sudhe\AppData\Local\Temp\ipykernel_23612\2965923628.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.in_memory import InMemoryDocstore


In [16]:
len(embeddings.embed_query("hello"))

3072

In [17]:
index = faiss.IndexFlatL2(3072) # 3072 is the dimension of the embeddings


In [18]:
# This is my vector  store
#  inside this vectorstore the data  is being stor insise the index
#  As of now it is inmomory but we can also store it in disk as well

vector_store = FAISS(embedding_function= embeddings,
                     index=index,
                     docstore= InMemoryDocstore(),
                     index_to_docstore_id={}
                     )

In [19]:
vector_store.add_texts(["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."])

['b2c98d34-99ed-46f3-97c9-fbc32b88c55d',
 'bee5319d-2e94-44b9-9dfb-c7a1d688dd43',
 '798cc314-93bf-4dfd-99d1-6164b616c33a']

In [20]:
vector_store.index_to_docstore_id

{0: 'b2c98d34-99ed-46f3-97c9-fbc32b88c55d',
 1: 'bee5319d-2e94-44b9-9dfb-c7a1d688dd43',
 2: '798cc314-93bf-4dfd-99d1-6164b616c33a'}

In [25]:
faiss_index_id =2

In [26]:
docstore_id=vector_store.index_to_docstore_id[faiss_index_id]

In [27]:
vector_store.docstore.search(docstore_id)

Document(id='798cc314-93bf-4dfd-99d1-6164b616c33a', metadata={}, page_content='Narendra Modi is a prime minister of India.')

In [28]:
vector_store.docstore.search(docstore_id).page_content

'Narendra Modi is a prime minister of India.'

In [29]:
vector_store.docstore.search(docstore_id).metadata

{}

In [30]:
#by K  select top k result  from indexs 
vector_store.similarity_search("Who is a president of USA?", k=1)

[Document(id='bee5319d-2e94-44b9-9dfb-c7a1d688dd43', metadata={}, page_content='Donald Trump is a president of USA')]

In [31]:
vector_store.similarity_search("Who is a president of USA?", k=2)

[Document(id='bee5319d-2e94-44b9-9dfb-c7a1d688dd43', metadata={}, page_content='Donald Trump is a president of USA'),
 Document(id='b2c98d34-99ed-46f3-97c9-fbc32b88c55d', metadata={}, page_content='Washington is the capital of USA')]

In [71]:
# langchain -->  documents --> embeddings --> vector_store --. index --> faiss
# langchain --> everything to be the landchain documents


In [32]:
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)


In [33]:
documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [34]:
vector_store.add_documents(documents=documents)

['530b0c64-3817-4ea6-9768-50d5cc8682fb',
 '3054564a-57ce-4673-8dfd-cbd329297cc6',
 'df692d6f-2cd7-4627-aa43-2d60fca13ff0',
 '8d02141a-e737-4232-b5b6-a25d0c31ea6b',
 'dd25701a-e025-4b33-aaa2-dbdee4f25624',
 '944afcd9-fb83-442c-b4f7-d95abbe034cc',
 '17e7db5f-cf9c-435f-a85e-233882fc1ab4',
 '01b9de54-7dcd-48d5-bb6b-a75b00b925aa',
 '3b1ac64d-becd-4d63-a662-ecdddfb96fbe',
 'c377beea-60f3-4023-941b-47ebe781b240']

In [35]:
vector_store.similarity_search("LangChain provides abstractions to make working with LLMs easy", k=5)

[Document(id='df692d6f-2cd7-4627-aa43-2d60fca13ff0', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='01b9de54-7dcd-48d5-bb6b-a75b00b925aa', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='530b0c64-3817-4ea6-9768-50d5cc8682fb', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(id='3b1ac64d-becd-4d63-a662-ecdddfb96fbe', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='3054564a-57ce-4673-8dfd-cbd329297cc6', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.')]

In [36]:
vector_store.similarity_search("LangChain provides abstractions to make working with LLMs easy", k=5, filter={"source": "news"})

[Document(id='3b1ac64d-becd-4d63-a662-ecdddfb96fbe', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='3054564a-57ce-4673-8dfd-cbd329297cc6', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='8d02141a-e737-4232-b5b6-a25d0c31ea6b', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [37]:
# disk persistance  of the  vector store
vector_store.save_local("fiass_index")